In [ ]:
!pip install google-generativeai

In [ ]:
# Minimal + reliable: Main PDF + per-paper appendices -> JSON array -> CSV


import os, json, csv
# import google.generativeai as genai
from google import genai
import time

API_KEY = os.environ.get("GOOGLE_API_KEY", "INSERT_API_HERE")  # <-- set your key or use Runtime > Variables
MODEL   = "gemini-1.5-pro"

FIELDS = [
    "Author(s), List of all authors of the study, Verbatim text",
    'Verbatim Text of the authors',
    "Title, Full title of the paper, Verbatim text",
    'Verbatim Text of the title',
    "Journal, Name of the journal where it was published, Verbatim text",
    'Verbatim Text of the journal',
    "Year of publication, Verbatim text",
    "APA style citation of the paper",

    "Copy of the abstract or summary of the study, Verbatim text",
    'Extract all independent or treatment variables studied. Identify what was experimentally manipulated or treated as causal',
    'Quote the exact text of the treatment description.',

    "Outcome / Dependent variable(s), Main outcome(s) measured in the study",
    'Verbatim Text of the Outcome',

    "Extract all survey questions presented to participants, including response options or scales (e.g., Likert, numeric ranges, categorical choices). Return the verbatim text of each question and its response options exactly as written in the paper. For each item, output in this format: Outcome: [Outcome / Dependent variable(s), Main outcome(s) measured in the study]  [verbatim text of the question and its response options]. Separate each item with one blank line.",
    "Identify participant incentives or rewards. Include award amount or form (money, credit, lottery).",
    'Verbatim Text of the Incentive',

    "Study type / Methodology, Design (experiment, observational, survey, natural experiment, qualitative)",
    'Verbatim Text of the Study type',
    "Extract the section that describes statistical analyses or estimation equations. Summarize model structure and variables, explaining components. Quote verbatim if equations are shown.",
    'Verbatim Text of the Study Analysis',
    "Level of analysis, Unit of analysis (individual, group, platform, nation, etc.), Can be a summary but must include coefficient sizes ",
    'Verbatim Text of the Level of analysis',
    "Extract all main effects reported in the study, including effect sizes, coefficients, test statistics, and confidence intervals. ",
    'Verbatim Text of the Level of the Main Effects',
    "Statistical Power (paper text)",
    'Verbatim Text of the Level of the Statistical Power',
    "Identify variables tested as moderators or sources of heterogeneous effects (treatment heterogeneity, subgroup differences, interaction terms).",
    'Verbatim Text of the Level of the Moderators',
    "Moderation Results (paper text),For each test, output in this format: Test/Interaction: [description of how moderation or heterogeneity was analyzed, including variables tested and model specification if given] Result: [summary of findings, including direction, magnitude, and statistical significance if reported] Include only tests that were explicitly conducted and reported, not hypotheses or qualitative speculations. Separate each item with one blank line.",
    'Verbatim Text of the Level of the Moderation Results',
    "Extract all demographic variables reported about the study participants (e.g., age, gender, socioeconomic status, education, political leaning, race/ethnicity, nationality). For each, output in this format: Demographic: [variable name] Description: [verbatim or summarized information about the sample’s distribution, statistics, or categories as reported in the paper]. Separate each item with one blank line.",
    'Verbatim Text of the Level of the Moderation Demographics',

    "Recruitment source, How the sample was obtained (MTurk, Prolific, panel, representative survey)",
    'Verbatim Text of the Level of the Recruitment source',
    "Sample size, Number of participants/observations",
    'Verbatim Text of the Level of the Sample size',
    "Country or region where study was conducted",
    "Socio-cultural context, Language, trust in media, media freedom, collectivism/individualism, etc.",
    "Political context, Election cycle, polarization level, regulatory environment",
    "Platform / Technological context, Platform studied, major platform events, affordances active, feed algorithm type",
    "Temporal context, Year(s) of data collection, relevant world events (pandemic, elections, etc.)",
    "Moderators recommended for the future , Moderators authors suggested for the future",
    "Research context notes, Other contextual notes (e.g., lockdown, unique cultural event)",
    "Intervention-relevant insights, Notes on practical or policy implications drawn from findings",
]
N = len(FIELDS)

PROMPT = f"""
TASK & SCOPE
You will read ONE academic paper PDF plus any APPENDICES provided. Treat all uploaded files as the SAME study.
Your job is to extract the fields below and return ONLY a JSON array.

OUTPUT FORMAT (STRICT)
- Return a JSON array of length {N} with {N} STRING elements in EXACTLY this order:
{json.dumps(FIELDS, ensure_ascii=False, indent=2)}
- Values ONLY. NEVER echo or include the field/column names or labels.
- The first character must be '[' and the last must be ']'. No extra text, no markdown/code fences.
- Each element MUST be a string. Arrays/objects/null/booleans are not allowed.
- Newlines are allowed inside a string when needed (e.g., multiple items); CSV will quote them.

SOURCE USE
- Use ONLY the uploaded paper and its appendices/supplementary. No outside knowledge or web search.
- If truly absent after checking all sections (front matter, abstract, intro, methods, measures/instruments, results, tables, figures, notes, limitations, conclusions, appendices), write "NOT SPECIFIED".

VERBATIM POLICY (STRICT)
- For any field whose name STARTS with "Verbatim Text", copy the FULL text span from the paper/appendix that contains the answer.
- No paraphrase. No ellipses. Include full sentences/item wording, response options/scale anchors, and all numbers/units.
- Preserve punctuation/case/symbols. Normalize only by: (a) collapsing multiple spaces, (b) converting hard line breaks to single newlines, (c) escaping internal double quotes by doubling them.
- If wording is split across table/figure/appendix cells, transcribe exactly and consolidate in reading order.

NON-VERBATIM POLICY
- Provide concise but COMPLETE content drawn ONLY from the paper/appendices. If a question needs direct answers, give direct short answers.
- Use compact clauses separated by semicolons; avoid long explanations and definitions unless the field explicitly asks for them.
- Always include exact numbers/units/dates when reported (e.g., "0.86 SD", "−0.57 Likert points", "22.1 pp", "95% CI [..]", "adj. p=..", "Spring 2017").
- Do NOT invent statistics or power statements; if not printed, summarize without numbers or write "NOT SPECIFIED".

MULTI-ANSWER QUESTIONS
- If a question needs multiple answers, then seperate by a new line. For example, when a questions asks for the survey questions, it is a multiple answer question, so seperate the answers by new lines.

TABLE HARVESTING
- If results appear in tables, extract ALL main effects relevant to the outcomes (across columns/panels/specs).
- Format: [Outcome/Index]: coefficient, SE or CI (if given), p or stars, sample/spec label.
- Keep numbers exactly as printed (including signs and stars if shown).

MULTI-STUDY HANDLING
- If multiple studies/samples/rounds are reported, include ALL within the same cell, prefixed "Study 1:", "Study 2:", etc.
- Apply the same rule to the paired VERBATIM fields (copy the correct wording per study).

CONSISTENCY & SANITY CHECKS
- Keep the array length EXACTLY {N}; do not add/remove fields.
- Every element must be a STRING and MUST NOT include the field title/label.
- Use dates/units exactly as printed (e.g., "Spring 2017", not invented day/months).
- If both online-first and print years appear, use the journal’s print year; optionally append "(online first: YYYY-MM-DD)" if printed in the PDF.

SELF-REPAIR (IF NEEDED)
- If your first draft is not valid JSON, has the wrong number of elements, or any element is empty even though the paper contains content, silently FIX and return a correct JSON array of length {N} (strings only, same order). Do not add commentary.

Now produce the JSON array.
"""

client = genai.Client(api_key=API_KEY)

# genai.configure(api_key=API_KEY)
#model = genai.GenerativeModel(MODEL)


def wait_until_active(uploaded_file, timeout=60):
    """Waits until the uploaded file is ACTIVE."""
    start_time = time.time()
    while time.time() - start_time < timeout:
        file_info = client.files.get(name=uploaded_file.name)
        if file_info.state == "ACTIVE":
            return file_info
        time.sleep(1)
    raise TimeoutError(f"File {uploaded_file.name} did not become ACTIVE within {timeout} seconds.")


def extract_row_array(pdf_path: str, appendix_paths=None):
    """Returns a Python list[str] of length N from the model (main PDF + optional appendices)."""

    appendix_paths = appendix_paths or []

    # Upload main paper + appendices
    main_file = wait_until_active(client.files.upload(file=pdf_path))
    appendix_files = [wait_until_active(client.files.upload(file=p)) for p in appendix_paths]

    # Send one user message containing prompt + all files
    parts = [PROMPT, main_file] + appendix_files

    result = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[
        parts],
    )
    #resp = model.generate_content(parts)
    text = (result.text or "").strip()

    # Strip accidental code fences if present
    if text.startswith("```"):
        text = text.strip("` \n")

    # Parse JSON (with fallback bracket slice)
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("["), text.rfind("]")
        data = json.loads(text[start:end+1])

    if not isinstance(data, list):
        raise ValueError("Model did not return a JSON array.")

    # Coerce to strings, enforce exact length
    data = ["" if v is None else str(v) for v in data]
    if len(data) < N:
        data += ["NOT SPECIFIED"] * (N - len(data))
    elif len(data) > N:
        data = data[:N]

    # Compact to single-line style
    data = [" ".join(s.split()) if isinstance(s, str) else "NOT SPECIFIED" for s in data]
    return data

def pdfs_to_csv(paper_paths, appendix_map, out_csv="/content/papers_extracted.csv"):
    """paper_paths: list[str]
       appendix_map: dict[str, list[str]] mapping each paper to its appendices (can be empty list)."""
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        writer.writerow(FIELDS)
        for p in paper_paths:
            print("Processing:", os.path.basename(p))
            row = extract_row_array(p, appendix_map.get(p, []))
            writer.writerow(row)
    print("Saved ->", out_csv)


In [ ]:
from google.colab import files
import os

# 1) Upload main papers
main_upload = files.upload()  # select one or more PDFs
paper_paths = ["/content/" + k for k in main_upload.keys()]

# 2) (Optional) For each paper, upload its appendices
appendix_map = {}
for p in paper_paths:
    yn = input(f"Upload appendices for {os.path.basename(p)}? (y/n): ").strip().lower()
    if yn.startswith("y"):
        up = files.upload()  # select one or more appendix files for THIS paper
        appendix_map[p] = ["/content/" + k for k in up.keys()]
    else:
        appendix_map[p] = []

# 3) Run extraction -> CSV
out_csv = "/content/papers_extracted.csv"
pdfs_to_csv(paper_paths, appendix_map, out_csv=out_csv)

# 4) Download CSV (optional)
files.download(out_csv)


Saving facebook_field_experiment_2024_u_s_.pdf to facebook_field_experiment_2024_u_s_.pdf
Upload appendices for facebook_field_experiment_2024_u_s_.pdf? (y/n): n
Processing: facebook_field_experiment_2024_u_s_.pdf
Saved -> /content/papers_extracted.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>